# SEOE — Sistema Especialista de Orientação de Estudos

Demonstração acadêmica da integração entre uma árvore de decisão e regras explícitas de sistema especialista. Todos os dados são fictícios e os resultados não representam validação pedagógica em estudantes reais.

In [ ]:
from pathlib import Path
import sys

raizProjeto = Path.cwd()
if not (raizProjeto / 'src').exists():
    raizProjeto = raizProjeto.parent

sys.path.insert(0, str(raizProjeto))

from matplotlib import pyplot as plt

from src.application.processamento_lote import processarAnaliseEmLote
from src.infrastructure.avaliacao import calcularAcuracia, calcularMatrizConfusao, extrairRegrasAprendidas
from src.infrastructure.dataset import carregarDataset
from src.infrastructure.treinamento import criarClassificador, dividirTreinoTeste, separarAtributosRotulo, treinarModelo
from src.presentation.entrada import criarConteudos
from src.presentation.graficos import gerarGraficoAcertos, gerarGraficoArvore, gerarGraficoAtencao, gerarGraficoHorasEstudadas
from src.presentation.relatorio import gerarRelatorioEspecifico, gerarRelatorioGeral

In [ ]:
dataset = carregarDataset(raizProjeto / 'data' / 'dataset_estudos.csv')
atributos, rotulos = separarAtributosRotulo(dataset)
particao = dividirTreinoTeste(atributos, rotulos)
classificador = treinarModelo(
    criarClassificador(),
    particao.atributosTreino,
    particao.rotulosTreino,
)
rotulosPrevistos = classificador.predict(particao.atributosTeste)

print(f'Acurácia: {calcularAcuracia(particao.rotulosTeste, rotulosPrevistos):.2%}')
display(calcularMatrizConfusao(particao.rotulosTeste, rotulosPrevistos))
print(extrairRegrasAprendidas(classificador))

## Casos demonstrativos

D1 apresenta baixo desempenho e prioridade alta; D2 apresenta desempenho elevado; D3 não possui questões e deve receber tratamento de dados insuficientes.

In [ ]:
registros = [
    {
        'disciplina': 'Português',
        'conteudo': 'Interpretação de texto',
        'horasEstudo': 3.0,
        'questoesRealizadas': 20,
        'questoesCorretas': 8,
        'dificuldadeDeclarada': 'sim',
        'prioridadeDeclarada': 'alta',
    },
    {
        'disciplina': 'Matemática',
        'conteudo': 'Porcentagem',
        'horasEstudo': 5.0,
        'questoesRealizadas': 30,
        'questoesCorretas': 24,
        'dificuldadeDeclarada': 'nao',
        'prioridadeDeclarada': 'media',
    },
    {
        'disciplina': 'História',
        'conteudo': 'Brasil Colônia',
        'horasEstudo': 2.0,
        'questoesRealizadas': 0,
        'questoesCorretas': 0,
        'dificuldadeDeclarada': 'sim',
        'prioridadeDeclarada': 'alta',
    },
]

conteudos = criarConteudos(registros)
relatorio = processarAnaliseEmLote(
    conteudos,
    classificador,
    particao.atributosTreino,
)

print(gerarRelatorioEspecifico(relatorio.resultadosEspecificos))
print()
print(gerarRelatorioGeral(relatorio.analiseGeral))

In [ ]:
gerarGraficoAcertos(relatorio.resultadosEspecificos)
gerarGraficoHorasEstudadas(relatorio.resultadosEspecificos)
gerarGraficoAtencao(relatorio.analiseGeral)
gerarGraficoArvore(classificador)
plt.show()

## Análise crítica

Com a divisão reprodutível, o modelo alcança 95% de acurácia no dataset fictício. Isso é esperado porque os rótulos foram definidos por critérios explícitos e simulados; portanto, o resultado demonstra a integração técnica entre dados, modelo e regras, mas não comprova eficácia educacional real.

A regra RE0 impede classificar conteúdos sem questões. Além disso, o ranking informa quando a amostra é pequena e alerta quando uma entrada está fora das faixas observadas no treino.